In [196]:
from pathlib import Path
import sys

print(sys.executable)
print(Path.cwd())

c:\dev\llm-data-analysis-study\.venv\Scripts\python.exe
c:\dev\llm-data-analysis-study\chapter03


In [197]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


def find_project_root(start: Path) -> Path:
    """현재 위치에서 위로 올라가며 data/raw 폴더가 있는 프로젝트 루트를 찾습니다."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists() and (candidate / "book").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. 노트북을 저장소 안에서 실행해 주세요.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR)


프로젝트 루트: C:\dev\llm-data-analysis-study
데이터 폴더: C:\dev\llm-data-analysis-study\data\raw


In [198]:
expected_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]

file_check = pd.DataFrame({
    "file": expected_files,
    "path": [str(DATA_DIR / filename) for filename in expected_files],
    "exists": [(DATA_DIR / filename).exists() for filename in expected_files],
})

file_check


,file,path,exists
0,customers.csv,C:\dev\llm-data-analysis-study\data\raw\custom...,True
1,products.csv,C:\dev\llm-data-analysis-study\data\raw\produc...,True
2,orders.csv,C:\dev\llm-data-analysis-study\data\raw\orders...,True
3,order_items.csv,C:\dev\llm-data-analysis-study\data\raw\order_...,True


In [199]:
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

print("customers:", type(customers))
print("products:", type(products))
print("orders:", type(orders))
print("order_items:", type(order_items))


customers: <class 'pandas.DataFrame'>
products: <class 'pandas.DataFrame'>
orders: <class 'pandas.DataFrame'>
order_items: <class 'pandas.DataFrame'>


In [200]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

list(datasets.keys())


['customers', 'products', 'orders', 'order_items']

분석할 데이터셋이 여러 개라서 딕셔너리로 묶어둔 것.

## 5. 데이터 크기 확인하기

`shape`는 데이터의 행과 열 개수를 알려 줍니다.

- 앞 숫자: 행 개수
- 뒤 숫자: 열 개수

예를 들어 `(150, 6)`은 150행 6열이라는 뜻입니다.


In [201]:
print("customers:", customers.shape)
print("products:", products.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)


customers: (150, 6)
products: (100, 4)
orders: (300, 5)
order_items: (764, 5)


### 생각해 보기

- 가장 행이 많은 데이터셋은 무엇인가요? order_items
- `order_items`가 `orders`보다 행이 많다면, 그 이유는 무엇일까요? 
orders는 주문 하나당 1개의 행을 차지하지만, order_items는 주문 하나에 포함된 1종류의 상품이 1개의 행을 차지하기 때문. 
- 분석 보고서에 데이터 규모를 설명한다면 어떤 문장으로 쓸 수 있을까요?
전체 데이터는 고객 150명, 상품 100종, 주문 300건으로 구성된다. 각 주문에 포함된 개별 상품 내역을 담은 주문 상세는 764건이다.

In [202]:
shape_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
    }
    for name, df in datasets.items()
])

shape_summary


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


## 6. 데이터 앞부분과 마지막 부분 보기

`head()`는 앞부분 5행을 보여 줍니다. 컬럼명이 예상과 맞는지, 값의 형태가 자연스러운지 빠르게 확인할 때 사용합니다.


In [203]:
customers.head()

,customer_id,name,gender,age,city,signup_date
0,1,김수민,F,19,광주,2024-08-29
1,2,김정호,F,32,대구,2026-01-12
2,3,이경수,F,61,성남,2024-08-22
3,4,조영호,F,55,울산,2026-06-23
4,5,이예원,F,19,부산,2024-11-23


In [204]:
products.head()

,product_id,product_name,category,price
0,1,전자기기 상품 001,전자기기,160000
1,2,도서 상품 002,도서,34000
2,3,전자기기 상품 003,전자기기,152000
3,4,생활용품 상품 004,생활용품,70000
4,5,식품 상품 005,식품,186000


In [205]:
orders.head()

,order_id,customer_id,order_date,payment_method,order_status
0,1,123,2026-07-17,card,completed
1,2,77,2025-10-02,naver_pay,cancelled
2,3,138,2026-01-29,bank_transfer,cancelled
3,4,57,2026-04-11,kakao_pay,cancelled
4,5,125,2026-03-02,card,cancelled


In [206]:
order_items.head()

,order_item_id,order_id,product_id,quantity,unit_price
0,1,1,100,3,102000
1,2,1,87,5,25000
2,3,1,7,3,142000
3,4,1,9,3,193000
4,5,2,72,4,189000


In [207]:
customers.tail()

,customer_id,name,gender,age,city,signup_date
145,146,김숙자,M,61,성남,2026-03-04
146,147,이정남,M,19,부산,2025-04-23
147,148,오도현,M,29,고양,2026-08-25
148,149,김정자,M,20,부산,2024-12-29
149,150,조미영,M,40,대전,2026-02-13


In [208]:
products.tail()

,product_id,product_name,category,price
95,96,생활용품 상품 096,생활용품,112000
96,97,전자기기 상품 097,전자기기,154000
97,98,스포츠 상품 098,스포츠,10000
98,99,뷰티 상품 099,뷰티,200000
99,100,도서 상품 100,도서,102000


In [209]:
orders.tail()

,order_id,customer_id,order_date,payment_method,order_status
295,296,116,2026-04-14,card,completed
296,297,22,2026-03-27,bank_transfer,completed
297,298,64,2026-03-06,naver_pay,cancelled
298,299,135,2025-10-18,bank_transfer,completed
299,300,96,2026-05-07,kakao_pay,cancelled


In [210]:
order_items.tail()

,order_item_id,order_id,product_id,quantity,unit_price
759,760,297,42,4,28000
760,761,298,40,4,174000
761,762,299,8,2,189000
762,763,299,12,4,175000
763,764,300,59,5,32000


### 생각해 보기

`head()`와 `tail()`을 보면서 아래를 확인해 보세요.

- 날짜처럼 보이는 컬럼이 있나요? 
customers 파일의 'signup date', orders 파일의 'order_date'
- 숫자처럼 보이는 컬럼이 있나요? 
- ID처럼 보이는 컬럼이 있나요? 
customer_id, product_id, order_id, order_item_id
- 사람이 직접 읽을 수 있는 이름이나 범주 값이 있나요?



## 7. 컬럼명 확인하기

컬럼명은 코드 작성에서 매우 중요합니다. 실제 컬럼명이 `customer_id`인데 LLM이나 사람이 `cust_id`라고 쓰면 코드는 실행되지 않습니다.


In [211]:
for name, df in datasets.items():
    print(f"[{name}]")
    print(list(df.columns))
    print()


[customers]
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
['product_id', 'product_name', 'category', 'price']

[orders]
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



In [212]:
column_summary = pd.DataFrame([
    {
        "dataset": name,
        "column_count": len(df.columns),
        "column_names": ", ".join(df.columns),
    }
    for name, df in datasets.items()
])

column_summary


,dataset,column_count,column_names
0,customers,6,"customer_id, name, gender, age, city, signup_date"
1,products,4,"product_id, product_name, category, price"
2,orders,5,"order_id, customer_id, order_date, payment_met..."
3,order_items,5,"order_item_id, order_id, product_id, quantity,..."


### 생각해 보기

- 고객을 구분하는 컬럼은 무엇인가요? 
- 주문을 구분하는 컬럼은 무엇인가요? 
- 상품을 구분하는 컬럼은 무엇인가요?
- 여러 파일을 연결할 때 사용할 수 있을 것 같은 컬럼은 무엇인가요? customer_id, product_id, order_id

## 8. 데이터 타입 확인하기

`info()`는 컬럼별 데이터 타입과 비어 있지 않은 값의 개수를 보여 줍니다.

특히 날짜처럼 보이지만 `object`로 저장된 컬럼을 주의해서 봅니다. pandas에서 `object`는 보통 문자열 또는 여러 타입이 섞인 컬럼일 때 나타납니다.


In [213]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 11.0 KB


In [214]:
for name, df in datasets.items():
    print(f"\n===== {name} =====")
    df.info()


===== customers =====
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 11.0 KB

===== products =====
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    100 non-null    int64
 1   product_name  100 non-null    str  
 2   category      100 non-null    str  
 3   price         100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 6.0 KB

===== orders =====
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns)

In [215]:
dtype_summary = pd.concat(
    [df.dtypes.rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("")

dtype_summary


,customers,products,orders,order_items
customer_id,int64,,int64,
name,str,,,
gender,str,,,
age,int64,,,
city,str,,,
signup_date,str,,,
product_id,,int64,,int64
product_name,,str,,
category,,str,,
price,,int64,,


### 생각해 보기

- 숫자형 컬럼은 어떤 것들이 있나요?
- 문자형 컬럼은 어떤 것들이 있나요?
- 날짜처럼 보이지만 아직 문자열일 가능성이 있는 컬럼은 무엇인가요? signup_date, order_date


## 9. 결측치 확인하기

결측치는 값이 비어 있는 상태입니다. 결측치가 있으면 평균, 비율, 그룹별 집계 결과가 달라질 수 있습니다.

`isna().sum()`은 컬럼별 결측치 개수를 계산합니다.


In [216]:
customers.isna().sum()

customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

In [217]:
missing_summary = pd.concat(
    [df.isna().sum().rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("").astype(str)

missing_summary


,customers,products,orders,order_items
customer_id,0.0,,0.0,
name,0.0,,,
gender,0.0,,,
age,0.0,,,
city,0.0,,,
signup_date,0.0,,,
product_id,,0.0,,0.0
product_name,,0.0,,
category,,0.0,,
price,,0.0,,


위는 결측치 개수 확인

In [218]:
missing_rate_summary = pd.concat(
    [(df.isna().mean() * 100).round(2).rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("")

missing_rate_summary

,customers,products,orders,order_items
customer_id,0.0,,0.0,
name,0.0,,,
gender,0.0,,,
age,0.0,,,
city,0.0,,,
signup_date,0.0,,,
product_id,,0.0,,0.0
product_name,,0.0,,
category,,0.0,,
price,,0.0,,


이건 결측치 비율 확인

In [219]:
duplicate_rows = pd.DataFrame([
    {
        "dataset": name,
        "duplicated_rows": df.duplicated().sum(),
    }
    for name, df in datasets.items()
])

duplicate_rows


,dataset,duplicated_rows
0,customers,0
1,products,0
2,orders,0
3,order_items,0


중복 확인 = 모든 컬럼 값이 완전히 동일한 행을 찾는 것.

In [220]:
id_duplicate_checks = pd.DataFrame([
    {
        "check": "customers.customer_id",
        "duplicated_count": customers["customer_id"].duplicated().sum(),
        "interpretation": "0이어야 고객 ID가 유일합니다.",
    },
    {
        "check": "products.product_id",
        "duplicated_count": products["product_id"].duplicated().sum(),
        "interpretation": "0이어야 상품 ID가 유일합니다.",
    },
    {
        "check": "orders.order_id",
        "duplicated_count": orders["order_id"].duplicated().sum(),
        "interpretation": "0이어야 주문 ID가 유일합니다.",
    },
    {
        "check": "order_items.order_id",
        "duplicated_count": order_items["order_id"].duplicated().sum(),
        "interpretation": "한 주문에 여러 상품이 있으면 0보다 클 수 있습니다.",
    },
])

id_duplicate_checks


,check,duplicated_count,interpretation
0,customers.customer_id,0,0이어야 고객 ID가 유일합니다.
1,products.product_id,0,0이어야 상품 ID가 유일합니다.
2,orders.order_id,0,0이어야 주문 ID가 유일합니다.
3,order_items.order_id,464,한 주문에 여러 상품이 있으면 0보다 클 수 있습니다.


### 생각해 보기

- `customers.customer_id` 중복과 `order_items.order_id` 중복은 왜 의미가 다를까요?
`customers.customer_id`는 각 고객에게 부여된 고유한 값이기 때문에 같은 값이 2번 나오는 것이 불가능하지만, `order_items.order_id`는 한 주문에 여러 상품이 포함될 수 있기 때문에, 한 상품이 각기 다른 주문에서 여러 번 반복되어 나올 수 있다.=즉 order_items에서 상품들을 쭉 나열했을 때 같은 order_id가 반복되어 나온다. 여기서는 orders가 총 300건이었고, order_items는 총 764건이므로, order_id가 464번 더 반복되어 나왔다는 결론. 
- 중복 개수만 보고 삭제하면 위험한 이유는 무엇일까요?
`order_items.order_id`같은 정상적인 반복을 중복으로 착각할 수 있기 때문에, 반복되어도 괜찮은 컬럼인지 확인하고 삭제해야 한다. 또 행 전체가 동일한 완전 중복도 다른 사건인데 우연히 값이 같은 경우가 있을 수 있다. 중복 제거 전에 원인을 반드시 파악하고 지워야, 데이터 수집 과정이나 시스템 버그상의 문제를 확인할 수 있다.

In [221]:
customers.describe()

,customer_id,age
count,150.000000,150.000000
mean,75.500000,42.086667
std,43.445368,15.613166
min,1.000000,19.000000
25%,38.250000,29.000000
50%,75.500000,40.000000
75%,112.750000,57.000000
max,150.000000,69.000000


In [222]:
products[["price"]].describe()

,price
count,100.000000
mean,110040.000000
std,56433.910574
min,5000.000000
25%,65750.000000
50%,112000.000000
75%,161000.000000
max,200000.000000


In [223]:
order_items[["quantity", "unit_price"]].describe()

,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


In [224]:
price_text = pd.Series(["10,000", "25,500", "3000", "확인필요"])
price_number = pd.to_numeric(
    price_text.str.replace(",", "", regex=False),
    errors="coerce",
)

pd.DataFrame({
    "original": price_text,
    "converted": price_number,
})


,original,converted
0,"10,000",10000.0
1,"25,500",25500.0
2,3000,3000.0
3,확인필요,NaN


In [225]:
customers["city"].value_counts().head(10)

city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

In [226]:
products["category"].value_counts()

category
스포츠     19
전자기기    17
생활용품    16
뷰티      16
도서      14
패션      11
식품       7
Name: count, dtype: int64

In [227]:
orders["order_status"].value_counts()


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

In [228]:
categorical_summary = pd.DataFrame([
    {"dataset": "customers", "column": "city", "unique_count": customers["city"].nunique()},
    {"dataset": "customers", "column": "gender", "unique_count": customers["gender"].nunique()},
    {"dataset": "products", "column": "category", "unique_count": products["category"].nunique()},
    {"dataset": "orders", "column": "payment_method", "unique_count": orders["payment_method"].nunique()},
    {"dataset": "orders", "column": "order_status", "unique_count": orders["order_status"].nunique()},
])

categorical_summary


,dataset,column,unique_count
0,customers,city,10
1,customers,gender,2
2,products,category,7
3,orders,payment_method,4
4,orders,order_status,3


.nunique() -> 그 컬럼에 몇 가지 서로 다른 (고유한) 값이 있는지 세는 함수. 즉 customers.city의 unique_count가 10이라는 건 10가지 도시 종류 있다는 것.

### 생각해 보기

- 특정 값에 데이터가 지나치게 몰려 있나요?
그렇지 않음.
- 오타나 표기 차이처럼 보이는 값이 있나요?
없음.
- 나중에 그룹별 분석 기준으로 쓰기 좋은 컬럼은 무엇인가요?
customers.city, products.category


## 13. 날짜 컬럼 확인하기

날짜 컬럼은 월별, 요일별, 기간별 분석에 자주 사용됩니다.

하지만 CSV에서 읽어온 날짜는 처음에는 문자열(`object`)일 수 있습니다. `pd.to_datetime()`으로 날짜 타입으로 바꿔야 날짜 계산을 안전하게 할 수 있습니다.


In [229]:
orders["order_date"].head()

0    2026-07-17
1    2025-10-02
2    2026-01-29
3    2026-04-11
4    2026-03-02
Name: order_date, dtype: str

In [230]:
print("변환 전 타입:", orders["order_date"].dtype)

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

print("변환 후 타입:", orders["order_date"].dtype)
print("날짜 변환 실패 건수:", orders["order_date"].isna().sum())
print("가장 빠른 주문일:", orders["order_date"].min())
print("가장 최근 주문일:", orders["order_date"].max())


변환 전 타입: str
변환 후 타입: datetime64[us]
날짜 변환 실패 건수: 0
가장 빠른 주문일: 2025-09-18 00:00:00
가장 최근 주문일: 2026-09-17 00:00:00


In [231]:

orders.assign(
    order_year=orders["order_date"].dt.year,
    order_month=orders["order_date"].dt.month,
    order_day_name=orders["order_date"].dt.day_name(),
).head()

,order_id,customer_id,order_date,payment_method,order_status,order_year,order_month,order_day_name
0,1,123,2026-07-17,card,completed,2026,7,Friday
1,2,77,2025-10-02,naver_pay,cancelled,2025,10,Thursday
2,3,138,2026-01-29,bank_transfer,cancelled,2026,1,Thursday
3,4,57,2026-04-11,kakao_pay,cancelled,2026,4,Saturday
4,5,125,2026-03-02,card,cancelled,2026,3,Monday


### 생각해 보기

- 데이터는 어느 기간을 포함하고 있나요?
2025-09-18 ~ 2026-09-17
- 월별 매출 분석을 하기에 충분한 기간인가요?
그렇다.
- 날짜 변환 실패 건수가 0보다 크다면 무엇을 확인해야 할까요?
날짜 형식이 이상했는지, 결측치인지, 오타나 잘못된 값이었는지 확인.
몇 건, 전체의 몇 %가 실패했는지 확인.
실패 건들이 특정 패턴을 갖는지 확인.



## 14. 여러 파일의 관계 확인하기

온라인 쇼핑몰 데이터는 고객, 상품, 주문, 주문 상세 데이터가 서로 연결되어야 분석할 수 있습니다.

| 연결 관계 | 의미 |
| --- | --- |
| `customers.customer_id` ↔ `orders.customer_id` | 어떤 고객이 주문했는지 연결합니다. |
| `orders.order_id` ↔ `order_items.order_id` | 주문과 주문 상세를 연결합니다. |
| `products.product_id` ↔ `order_items.product_id` | 주문 상세와 상품 정보를 연결합니다. |

In [232]:
invalid_customers = orders[~orders["customer_id"].isin(customers["customer_id"])]
invalid_orders = order_items[~order_items["order_id"].isin(orders["order_id"])]
invalid_products = order_items[~order_items["product_id"].isin(products["product_id"])]

relationship_check = pd.DataFrame([
    {
        "relationship": "orders.customer_id -> customers.customer_id",
        "invalid_rows": len(invalid_customers),
    },
    {
        "relationship": "order_items.order_id -> orders.order_id",
        "invalid_rows": len(invalid_orders),
    },
    {
        "relationship": "order_items.product_id -> products.product_id",
        "invalid_rows": len(invalid_products),
    },
])

relationship_check


,relationship,invalid_rows
0,orders.customer_id -> customers.customer_id,0
1,order_items.order_id -> orders.order_id,0
2,order_items.product_id -> products.product_id,0


invalid_rows가 모두 0이므로 샘플 데이터에서는 기본적인 연결 관계가 유지되고 있다는 것. 0보다 큰 값이 있으면 어느 파일에서 기준 ID가 빠져 있는지 확인.

## 15. 간단한 병합으로 관계 확인하기

키 관계가 맞는지 확인한 뒤에는 데이터를 병합해 볼 수 있습니다. 아래 코드는 주문 상세(`order_items`)에 상품 정보(`products`)를 붙이고, 각 행의 금액을 계산합니다.


In [233]:
order_items_with_products = order_items.merge(
    products,
    on="product_id",
    how="left",
)

order_items_with_products["line_amount"] = (
    order_items_with_products["quantity"] * order_items_with_products["unit_price"]
)

order_items_with_products.head()

,order_item_id,order_id,product_id,quantity,unit_price,product_name,category,price,line_amount
0,1,1,100,3,102000,도서 상품 100,도서,102000,306000
1,2,1,87,5,25000,도서 상품 087,도서,25000,125000
2,3,1,7,3,142000,도서 상품 007,도서,142000,426000
3,4,1,9,3,193000,스포츠 상품 009,스포츠,193000,579000
4,5,2,72,4,189000,뷰티 상품 072,뷰티,189000,756000


In [234]:
category_sales = (
    order_items_with_products
    .groupby("category", as_index=False)["line_amount"]
    .sum()
    .sort_values("line_amount", ascending=False)
)

category_sales

,category,line_amount
3,스포츠,50174000
1,뷰티,47551000
5,전자기기,41003000
2,생활용품,34839000
4,식품,33597000
0,도서,24645000
6,패션,23801000


## 16. 반복 점검을 함수로 정리하기

여러 데이터셋에 같은 점검을 반복할 때는 함수로 정리하면 편합니다.


In [235]:
def check_data_overview(name: str, df: pd.DataFrame) -> None:
    print(f"===== {name} =====")
    print("shape:", df.shape)
    print("\ncolumns:")
    print(list(df.columns))
    print("\ndtypes:")
    print(df.dtypes)
    print("\nmissing values:")
    print(df.isna().sum())
    print("\nduplicated rows:", df.duplicated().sum())


check_data_overview("customers", customers)


===== customers =====
shape: (150, 6)

columns:
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

dtypes:
customer_id    int64
name             str
gender           str
age            int64
city             str
signup_date      str
dtype: object

missing values:
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

duplicated rows: 0


In [236]:
for name, df in datasets.items():
    check_data_overview(name, df)
    print()

===== customers =====
shape: (150, 6)

columns:
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

dtypes:
customer_id    int64
name             str
gender           str
age            int64
city             str
signup_date      str
dtype: object

missing values:
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

duplicated rows: 0

===== products =====
shape: (100, 4)

columns:
['product_id', 'product_name', 'category', 'price']

dtypes:
product_id      int64
product_name      str
category          str
price           int64
dtype: object

missing values:
product_id      0
product_name    0
category        0
price           0
dtype: int64

duplicated rows: 0

===== orders =====
shape: (300, 5)

columns:
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

dtypes:
order_id                   int64
customer_id                int64
order_date        datetime64[us]
payment_method          

## 17. LLM에게 데이터 구조를 설명시키는 법

LLM에게 원본 데이터를 그대로 붙여 넣는 것은 피하는 것이 좋습니다. 대신 아래처럼 구조 요약만 전달합니다.

- 파일명
- 컬럼명
- 행과 열 개수
- 데이터 타입
- 결측치 개수
- 중복 여부
- 파일 간 키 관계


In [237]:
llm_dataset_summary = shape_summary.merge(column_summary, on="dataset")
llm_dataset_summary


,dataset,rows,columns,column_count,column_names
0,customers,150,6,6,"customer_id, name, gender, age, city, signup_date"
1,products,100,4,4,"product_id, product_name, category, price"
2,orders,300,5,5,"order_id, customer_id, order_date, payment_met..."
3,order_items,764,5,5,"order_item_id, order_id, product_id, quantity,..."


In [238]:
for _, row in llm_dataset_summary.iterrows():
    print(f"- {row['dataset']}: {row['rows']}행 {row['columns']}열")
    print(f"  컬럼: {row['column_names']}")


- customers: 150행 6열
  컬럼: customer_id, name, gender, age, city, signup_date
- products: 100행 4열
  컬럼: product_id, product_name, category, price
- orders: 300행 5열
  컬럼: order_id, customer_id, order_date, payment_method, order_status
- order_items: 764행 5열
  컬럼: order_item_id, order_id, product_id, quantity, unit_price


## 18. LLM 답변 검증 연습

LLM이 다음과 같이 답했다고 가정해 봅니다.

> 고객 데이터에 age 컬럼이 있으므로 연령대별 매출 분석을 바로 수행하면 됩니다.

이 답변은 그럴듯하지만 충분히 안전하지 않습니다. 아래 내용을 직접 확인해야 합니다.

- `age` 컬럼이 실제로 존재하는가?
- `age` 컬럼에 결측치나 이상치가 있는가?
- 고객 데이터와 주문 데이터가 `customer_id`로 연결되는가?
- 매출을 계산하려면 주문 상세와 상품 또는 단가 정보가 필요한가?
- 취소 주문을 포함할지 제외할지 기준이 있는가?


In [239]:
validation_check = pd.DataFrame([
    {
        "question": "customers에 age 컬럼이 있는가?",
        "result": "age" in customers.columns,
    },
    {
        "question": "age 결측치 개수는?",
        "result": customers["age"].isna().sum() if "age" in customers.columns else "컬럼 없음",
    },
    {
        "question": "orders.customer_id가 customers.customer_id와 연결되는가?",
        "result": len(invalid_customers) == 0,
    },
    {
        "question": "매출 계산에 필요한 quantity와 unit_price가 있는가?",
        "result": {"quantity", "unit_price"}.issubset(order_items.columns),
    },
])

validation_check


,question,result
0,customers에 age 컬럼이 있는가?,True
1,age 결측치 개수는?,0
2,orders.customer_id가 customers.customer_id와 연결되는가?,True
3,매출 계산에 필요한 quantity와 unit_price가 있는가?,True


## 19. 이번 장 점검 체크리스트

| 점검 항목 | 확인 |
| --- | --- |
| 필요한 CSV 파일이 모두 존재하는가? | ✅ |
| 각 데이터셋의 행과 열 개수를 확인했는가? | ✅ |
| 컬럼명이 예상과 일치하는가? | ✅ |
| 날짜 컬럼의 데이터 타입을 확인했는가? | ✅ |
| 숫자 컬럼이 실제 숫자형으로 저장되어 있는가? | ✅ |
| 결측치가 있는 컬럼을 확인했는가? | ✅ |
| 중복 데이터가 있는지 확인했는가? | ✅ |
| 주요 ID 컬럼의 중복 여부를 확인했는가? | ✅ |
| 여러 파일을 연결할 키 컬럼을 확인했는가? | ✅ |
| 파일 간 키 관계가 실제로 연결 가능한지 확인했는가? | ✅ |
| LLM에 원본 데이터 대신 구조 요약만 입력했는가? | ✅ |
| LLM이 제안한 설명을 실제 데이터와 비교해 검증했는가? | ✅ |


## 20. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 4개 CSV 파일의 행과 열 개수를 하나의 표로 정리하세요.
2. 각 파일의 결측치 개수와 결측치 비율을 확인하세요.
3. `orders.order_date`를 날짜 타입으로 변환하고 데이터 기간을 확인하세요.
4. `order_items`와 `products`를 병합해 카테고리별 주문 금액을 계산하세요.
5. LLM에게 데이터 구조 요약을 전달하는 프롬프트를 직접 작성하세요.
6. LLM이 만든 분석 아이디어가 실제 컬럼과 키 관계에 맞는지 검증하세요.


In [240]:
# 1. 4개 CSV 파일의 행과 열 개수를 하나의 표로 정리하세요.

shape_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for name, df in datasets.items()
])

shape_summary

,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


In [241]:
# 2. 각 파일의 결측치 개수와 결측치 비율을 확인하세요.

missing_summary = pd.concat(
    [df.isna().sum().rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("").astype(str)

missing_summary

,customers,products,orders,order_items
customer_id,0.0,,0.0,
name,0.0,,,
gender,0.0,,,
age,0.0,,,
city,0.0,,,
signup_date,0.0,,,
product_id,,0.0,,0.0
product_name,,0.0,,
category,,0.0,,
price,,0.0,,


In [242]:
missing_rate_summary = pd.concat(
    [(df.isna().mean() * 100).round(2).rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("")

missing_rate_summary

,customers,products,orders,order_items
customer_id,0.0,,0.0,
name,0.0,,,
gender,0.0,,,
age,0.0,,,
city,0.0,,,
signup_date,0.0,,,
product_id,,0.0,,0.0
product_name,,0.0,,
category,,0.0,,
price,,0.0,,


In [243]:
# 3. `orders.order_date`를 날짜 타입으로 변환하고 데이터 기간을 확인하세요.

print("변환 전 타입:", orders["order_date"].dtype)

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

print("변환 후 타입:", orders["order_date"].dtype)
print("날짜 변환 실패 건수:", orders["order_date"].isna().sum())
print("가장 빠른 주문일:", orders["order_date"].min())
print("가장 최근 주문일:", orders["order_date"].max())



변환 전 타입: datetime64[us]
변환 후 타입: datetime64[us]
날짜 변환 실패 건수: 0
가장 빠른 주문일: 2025-09-18 00:00:00
가장 최근 주문일: 2026-09-17 00:00:00


In [244]:
# 4. `order_items`와 `products`를 병합해 카테고리별 주문 금액을 계산하세요.

category_sales = (
    order_items_with_products
    .groupby("category", as_index=False)["line_amount"]
    .sum()
    .sort_values("line_amount", ascending=False)
)

category_sales



,category,line_amount
3,스포츠,50174000
1,뷰티,47551000
5,전자기기,41003000
2,생활용품,34839000
4,식품,33597000
0,도서,24645000
6,패션,23801000


# 5. LLM에게 데이터 구조 요약을 전달하는 프롬프트를 직접 작성하세요.

- 파일명: customers.csv
- 컬럼명: customer_id, name, gender, age, city, signup_date
- 행과 열 개수: 150행 × 6열
- 데이터 타입: customer_id(int64), name(str), gender(str), age(int64), 
  city(str), signup_date(str)
- 결측치 개수: 전 컬럼 0건
- 중복 여부: customer_id 중복 0건, 전체 행 중복 0건
- 파일 간 키 관계: customer_id가 orders.customer_id의 참조 대상

- 파일명: products.csv
- 컬럼명: product_id, product_name, category, price 
- 행과 열 개수: 100행 × 4열
- 데이터 타입: product_id(int64), product_name(str), category(str), price(int64)
- 결측치 개수: 전 컬럼 0건
- 중복 여부: product_id 중복 0건, 전체 행 중복 0건
- 파일 간 키 관계: product_id가 order_items.product_id의 참조 대상

- 파일명: orders.csv
- 컬럼명: order_id, customer_id, order_date, payment_method, order_status
- 행과 열 개수: 300행 × 5열
- 데이터 타입: order_id(int64), customer_id(int64), order_date(str),   payment_method(str), order_status(str)
- 결측치 개수: 전 컬럼 0건
- 중복 여부: order_id 중복 0건, 전체 행 중복 0건
- 파일 간 키 관계: customer_id는 customers.customer_id를 참조, 
  order_id는 order_items.order_id의 참조 대상

  파일명: order_items.csv
- 컬럼명: order_item_id, order_id, product_id, quantity, unit_price
- 행과 열 개수: 764행 × 5열
- 데이터 타입: order_item_id(int64), order_id(int64), product_id(int64), quantity(int64), unit_price(int64)
  unit_price(int64)
- 결측치 개수: 전 컬럼 0건
- 중복 여부: 전체 행 중복 0건 / 단, order_id 값 자체는 464건 반복 
  (orders 대비 order_items 행이 더 많아서 발생하는 정상적인 1:N 구조이며, 
  오류가 아님)
- 파일 간 키 관계: order_id는 orders.order_id를 참조(자식 키), 
  product_id는 products.product_id를 참조(자식 키)


# 6. LLM이 만든 분석 아이디어가 실제 컬럼과 키 관계에 맞는지 검증하세요.
아래 마크다운에 작성.

# Chapter 03 제출 답안 양식. 데이터의 첫인상 읽기

## 0. 제출 정보
- 이름: 황지원
- GitHub ID: Jiwon0712
- 작성일: 2026.09.20
- 최종 제출 URL: 

## 1. 데이터 로딩과 구조 확인
### 실행/결과
- 4개 CSV 로딩 여부: 정상 로딩 완료
- 각 데이터 shape: customers(150,6), products(100,4), orders(300,5), order_items(764,5)
- 주요 컬럼: customer_id, product_id, order_id, order_date, quantity, unit_price 등
- dtypes에서 주목한 컬럼: signup_date, order_date가 str(문자열)로 저장되어 있어 datetime 변환 필요

### Evidence
![데이터 구조 확인](images/step01_structure.png)

### 결과 관찰
customers은 150명, 6개의 컬럼을 가진다. products는 100종, 4개의 컬럼을 가진다. orders는 300건, 5개의 컬럼을 가진다. order_items는 764개 행과 5개 컬럼을 가진다. 각 파일에서 겹치는 컬럼에는 customer_id, product_id, order_id가 있고, 이외에도 quantity, unit_price, price 등의 컬럼들이 있다. 

### 나의 해석과 판단
분석 전에 특히 주의해야 할 데이터셋/컬럼과 이유를 작성하세요.

customers/signup_date와 orders/order_date가 문자열로 저장되어 있는데, 날짜 분석하려면 datetime으로 전환할 필요가 있다. 


### 업무·분석적 의미
구조를 먼저 확인하지 않고 분석을 시작할 때 생길 수 있는 문제를 작성하세요.

실제로는 문자열인 컬럼을 숫자나 날짜로 착각해 잘못된 연산을 수행하거나, 컬럼명/데이터 형태를 오해해 잘못된 결론을 낼 위험이 있다.

### 한계와 추가 확인 사항
아직 알 수 없는 품질 문제를 작성하세요.

현재 shape와 dtype만으로는 값 자체의 논리적 타당성은 확인되지 않았으므로 추가 점검이 필요하다. 예를 들어 나이가 음수인지, 아니면 날짜가 미래의 날짜인지 등을 점검해야 한다.

## 2. 결측·중복·키 품질
- 주요 ID 결측: customer_id, product_id, order_id 모두 결측치 없음
- 주요 ID 중복: customer_id 0건, product_id 0건, order_id(orders) 0건, order_id(order_items) 464건
- 전체 행 중복: customers 0건, products 0건, orders 0건, order_items 0건

![결측 중복 점검](images/step02_quality.png)

### 결과 관찰

customer_id, product_id, order_id 모두 결측치는 없었다. 전체 행이 중복되는 것은 customers, products, orders, order_items 파일 모두 0건이었다. 주요 ID 중복에서는 order_id(order_items)만 464건 중복이 있었다. 

### 나의 해석과 판단
어떤 문제를 먼저 처리해야 하는지 우선순위를 작성하세요.

1. 주요 ID의 중복 여부 확인. 이 값들이 고유하지 않으면 이후의 모든 결과가 왜곡되기 때문에 가장 먼저 확인해야 함. customer_id, product_id, order_id(orders) 모두 중복 0건으로 정상.
2. 중복되는 값이 실제로 오류인지 확인. 삭제 여부를 잘못 판단할 시 유효한 데이터를 잘못 삭제할 수도 있기 때문에 오류 원인을 짚고 넘어가야 함. 여기서는 order_id(order_items)의 중복이 464건 집계되었는데, 한 주문당 상품이 여러 개 포함될 경우 order_id가 중복될 수 있기 때문에 오류가 아님.
3. 결측치 처리. 컬럼별로 결측치 비율을 계산하여 비율이 높은 컬럼부터 결측이 발생한 원인을 분석한다. 집계 결과 customer_id, product_id, order_id 모두 결측치 없음.

### 업무·분석적 의미

중복 개수만 보고 삭제하면, order_items의 order_id처럼 정상적으로 반복되는 값까지 오류로 오인해 유효한 데이터를 삭제할 위험이 있다. 

### 한계와 추가 확인 사항

빈값이나 가짜값으로 채워진 결측치가 있을 수도 있다. 예를 들어, 확인필요나, N/A 등의 문자열이 결측치 대신에 들어있을 수 있다. 또 논리적으로 이상한 값이 들어 있는 경우도 결측은 아니지만 걸러낼 필요가 있다. (예를 들어 age 값이 -5처럼 비논리적인 값이 들어 있을 수 있음.) 또 완전 중복은 0건으로 확인되긴 했지만, 철자나 표기만 다르게 표시되어 실제로는 같은 사건 혹은 고객이 중복된 경우가 있을 수도 있다.


## 3. 숫자형·범주형·날짜 점검
- 숫자형 범위에서 주목한 값: count(데이터 개수), mean(평균), std(표준편차), min(최솟값), max(최댓값), 25%(1사분위수), 50%(중앙값), 75%(3사분위수)
- 범주형 빈도에서 주목한 값: unique_count(몇 개의 범주가 있는지)
- 날짜 변환 실패 건수: 0건
- 날짜 범위: 2025-09-18 ~ 2026-09-17

![기본 분포와 날짜 확인](images/step03_distribution.png)

### 결과 관찰
- 숫자형 컬럼(age, price, quantity, unit_price)의 describe() 결과, 
  age는 최소 19세~최대 69세로 비현실적인 값은 발견되지 않았으며, 
 price는 최소 5000원~최대 200000원으로, quantity는 최소 1개~최대 5개로 특별한 극단값은 보이지 않았다. 
- 범주형 컬럼은 각각 합리적인 개수의 고유값을 가지고 있다 
  (city 10개, gender 2개, category 7개, payment_method 4개, order_status 3개).
- order_date를 datetime으로 변환한 결과 변환 실패 건수는 0건이다. order_date의 날짜 범위는 (2025-09-18 ~2026-09-17)로 확인되었다.

### 나의 해석과 판단
이상해 보이는 값이 실제 오류인지 업무적으로 가능한 값인지 구분하기 위해 무엇을 더 확인해야 하는지 작성하세요.

- age가 19~69세, price가 5,000원~200,000원, quantity가 1개~5개 범위에 
있다는 것만으로는 이 값이 정상 범위인지 업무 규칙을 벗어난 이상치인지 
판단할 수 없다. 예를 들어 200,000원짜리 상품이 실제 고가 상품 카테고리에 
속하는지, 아니면 입력 오류로 0이 하나 더 붙은 것인지는 price와 
category를 교차 확인해야 알 수 있다. 마찬가지로 quantity 최댓값 5도 
대량구매 고객의 정상 주문인지, 재고 관리 시스템의 제한값인지 구분이 필요하다.
- 범주형 값의 개수가 합리적으로 보이긴 하지만, 개수만으로는 서울과 seoul처럼 표기만 다른 동일 범주가 섞여 있는지 알 수 없으므로 실제 값 목록을 
직접 확인해야 한다. 물품의 카테고리는 직접 확인 결과 표기 오류나 동일 범주는 없었다.
- 날짜 변환 실패가 0건이고 범위가 2025-09-18~2026-09-17로 
약 1년치인 것은 형식상 정상이지만, 이 기간이 실제 서비스 운영 기간과 
일치하는지는 추가적인 확인이 필요하다. 

### 업무·분석적 의미

- 숫자형 값의 범위를 모른 채 분석을 시작하면, 이상치 하나가 평균이나 
합계 같은 집계 지표를 크게 왜곡시켜 잘못된 의사결정으로 이어질 수 있다. 
- 범주형 값에 표기 오류가 숨어 있는 채로 그룹별 집계를 하면, 실제로는 같은 도시인데 서로 다른 그룹으로 분리되어 지역별 매출 같은 분석 결과가 실제보다 분산되어 나타나는 문제가 생긴다. 
- 날짜 변환 실패를 무시하고 넘어가면, 이후 집계·시각화 단계에서 누락된 데이터로 인해 실제보다 축소된 결과가 나올 수 있으며, 특히 월별 추이 분석처럼 날짜를 기준으로 삼는 지표는 결과 자체가 왜곡될 위험이 크다.

### 한계와 추가 확인 사항

- 숫자형 범위에서 min과 max값만 보고 이상치가 없다고 판단하는 것은 위험하다. 사분위값들을 기준으로 이상치를 통계적으로 걸러내야 한다.
- 범주형 빈도에서 몇 가지 값이 있는지 개수를 세는 nunique는 실제로 이 값들이 무엇인지 판단하지 않는다. 즉 철자나 표기만 다를 뿐 의미상으로는 같은 범주로 세어야 되는 값이 있는지 판단이 필요하다. 예를 들어 서울과 seoul이 다른 범주로 카운트되었을 수 있다.
- 날짜 변환 실패 건수는 0건이지만 논리적으로 이상한 날짜는 아직 걸러내지 않았다.
- 날짜 범위가 실제로 회사 영업 시간 등과 일치하는지(업무적으로 말이 되는 기간인지) 판단이 필요하다.

## 4. CSV 간 키 관계 검증
- 없는 `customer_id`: 0건
- 없는 `order_id`: 0건
- 없는 `product_id`: 0건

![PK FK 관계 검증](images/step04_relationship.png)

### 결과 관찰

orders의 customer_id, order_items의 order_id와 product_id를 각각 
customers, orders, products의 ID와 대조한 결과, 세 항목 모두 
연결되지 않는 값이 0건으로 나타나 테이블 간 연결 관계가 정상적으로 
유지되고 있음을 확인하였다.

### 나의 해석과 판단
키 관계 문제가 발견되었다면 바로 삭제하면 안 되는 이유를 작성하세요.

연결 항목 불일치가 발견될 경우, 그 불일치하는 항목이 존재하는 이유가 데이터 이관 과정의 오류일 수도 있지만 탈퇴한 고객의 과거 주문 기록이나 단종된 상품의 과거 판매 내역처럼 정상적인 상황일 수도 있기 때문이다. 삭제 전에 해당 기록의 발생 시점과 다른 컬럼을 함께 확인해서 원인을 파악해야 한다. 

### 업무·분석적 의미

키 관계가 깨진 상태로 분석을 진행하면, merge 시 해당 행이 누락되거나 결측치로 처리되어 매출이나 고객 수 같은 집계 지표가 실제보다 축소되어 계산될 위험이 있다. 특히 정확한 숫자가 중요한 분석에서는 이런 누락이 의사결정에 직접적인 영향을 줄 수 있다.

### 한계와 추가 확인 사항

이번 검증은 ID 값이 다른 테이블에 존재하는지만 확인한 것이다. order_items의 product_id가 products 테이블에 존재하기만 하면 통과된 것으로 처리했는데, 실제로 그 product_id에 연결된 unit_price가 products에 등록된 price와 실제로 일치하는지, 혹은 order_items의 customer 정보가 orders를 거쳐 customers와 연결됐을 때 나이나 지역 같은 값이 서로 모순되지는 않는지는 확인하지 않았다.

## 5. LLM 구조 설명 검증
- LLM에 제공한 Safe Context: customers.info() 출력 결과(컬럼명, dtype, 
  non-null count), 각 데이터셋의 shape(150,6 / 100,4 / 300,5 / 764,5), 
  describe() 결과의 통계 요약값, categorical_summary 표(컬럼별 고유값 
  개수)
- LLM이 제안한 추가 점검: 
  1) order_items의 order_id 반복을 단순 중복으로 오인하지 말고, orders와의 
     1:N 관계 관점에서 확인할 것
  2) 중복 및 결측치 개수뿐 아니라 비율까지 함께 봐야 데이터셋 간 심각도를 
     비교할 수 있다는 점
  3) nunique()로 확인한 범주형 고유값 개수만으로는 표기 오류(예: 
     "서울" vs "서울특별시")를 잡아낼 수 없으니 value_counts()로 실제 
     값 목록을 확인할 것
  4) 키 관계(FK) 불일치가 발견되어도 바로 삭제하지 말고 원인(이관 오류/
     탈퇴 고객/단종 상품 등)을 먼저 구분할 것

- 실제 데이터에서 확인한 항목: order_items의 order_id 반복이 실제로 
  orders(300건)와 order_items(764건)의 차이(464건)와 정확히 일치함을 
  확인했다. 중복 및 결측치 비율 계산 결과 0%였다. products의 category의 범주형 값들을 실제로 확인한 결과 표기 오류나 동일 범주의 값은 없었다. 키 관계 불일치 건수도 isin()으로 직접 계산해 0건임을 확인했다.

- 채택/수정/보류한 내용: order_id 반복을 오류로 보지 말라는 제안은 
  실제 데이터 구조(1:N)와 정확히 맞아떨어져 그대로 채택했다. 중복 및 결측치는 0건으로 집계되어 비율은 간단하게 정리만 하였다. 
  value_counts()로 범주형 실제 값을 확인하라는 제안은 아직 일부 실행하지 
  못해 보류 상태이다. 탈퇴 고객이나 단종 상품을 고려하라는 제안은 
  현재 프로젝트 범위를 다소 벗어난다고 판단하였고, 또한 키 관계 불일치가 0건으로 나왔기에 참고만 하고 본문에는 간단히만 작성했다.

![LLM 구조 검토](images/step05_llm.png)

### 나의 해석과 판단
LLM 제안 중 가장 유용했던 것과 가장 조심해야 할 것을 작성하세요.

가장 유용했던 것은 order_items의 order_id 반복을 실제 오류로 판단하지 않도록 짚어준 부분이었다. 단순히 중복으로 판단했다면 유효한 데이터를 삭제했었을 것이다. 또 범주형 값들 중 표기 오류로 의미상 동일 범주인데 다르게 분류된 값이 있을 수 있으니 직접 목록을 확인하라는 제안도 유용했다. 

가장 조심해야 할 것은 LLM이 제안한 내용이 실제 내 데이터 값이나 컬럼명 등을 알고 제안한 것이 아니기 때문에 내가 전달한 요약 정보만을 근거로 일반적으로 발생하는 사안들만 짚어준다는 것이었다. 실제로는 결측치가 0건이었는데, LLM은 결측치가 존재한다는 가정 하에 다음 스텝을 제안했기 때문에 그대로 따랐다면 불필요한 과정을 거쳤을 것이다. 

### 한계와 추가 확인 사항
LLM에는 간단한 데이터 구조만 전달하였고 실제 개별 데이터 값의 이상치나 철자 오류까지는 짚어주지 못하였다. 원본 데이터를 직접 보지 않고는 판단할 수 없기 때문에, value_counts라든지 개별 행을 완전히 필터링하는 과정이라든지, 내가 직접 실행하여 검증하는 부분은 여전히 일부 남아 있다. 

## 6. Chapter 03 최종 판단
### 데이터의 첫인상 3가지
1. 데이터 상태가 양호한 것 같다. customers, products, orders, order_items 모두 결측치나 완전히 중복되는 행이 0건이었다. =
2. 날짜를 나타내는 컬럼이 문자열로 표시된 것을 보니, 데이터로서 아직 형식이 미비한 컬럼들이 일부 있다. 
3. order_items(764행)가 orders(300행)보다 행 수가 훨씬 많아, 테이블마다 하나의 행이 의미하는 단위가 다르다는 점을 알 수 있었다.

### 다음 Chapter 전에 반드시 확인/처리해야 할 항목
1. 범주형 컬럼 중 아직 확인하지 않은 name, city, payment_method, order_status, product_name의 실제 값 목록을 value_counts()로 직접 확인해, 표기만 다른 동일 범주가 섞여 있지 않은지 마저 검증해야 한다. 
2. 숫자형 컬럼인 price, quantity, unit_price의 이상치를 IQR 등 통계적 기준으로 한 번 더 점검해야 한다. 
3. price(products)와 unit_price(order_items) 두 컬럼이 실제로 같은지 확인한다. 

### 현재 데이터만으로 단정할 수 없는 것

결측치, 중복, 키 관계는 모두 정상으로 확인됐지만, 이는 이 데이터가 
실제로 정확하다는 것을 보장하지는 않는다. 예를 들어 price와 
unit_price 값 자체가 타당한 금액인지, order_status나 
payment_method의 각 값이 실제 운영 기준과 일치하는지는 이번 
구조 점검만으로는 알 수 없다. 이는 원본 시스템과의 
대조가 있어야 판단 가능한 부분이다.

## 최종 제출 체크
- [x] Notebook을 처음부터 끝까지 실행했습니다.
- [x] 오류 셀이 남아 있지 않습니다.
- [x] 핵심 Evidence를 첨부했습니다.
- [x] 관찰과 해석을 구분했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter03/chapter03.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.